In [5]:
import sys, os
sys.path.append(os.path.join('/home/module'))
from Bq_Connect import BQDataProcessor 


/opt/conda/lib/python3.8/site-packages/google/cloud/bigquery_storage_v1/__init__.py:57: FutureWarning: You are using a non-supported Python version (3.8.16).  Google will not post any further updates to google.cloud.bigquery_storage_v1 supporting this Python version. Please upgrade to the latest Python version, or at least to Python 3.9, and then update google.cloud.bigquery_storage_v1.
  warnings.warn(


In [6]:
sql = BQDataProcessor(env="dev") 

[BQDataProcessor] Connected → env=DEV, project=pgd-dev-data-analytics, SA=default (/home/jupyterhub/SA/jupyter-pgd-dev-data-analytics-296ca984dd2f.json)


In [25]:
SELECT_QUERY = """
WITH aktif_cif AS (
    SELECT DISTINCT 
        cif, 
        branch_code, 
        create_date -- This determines the type (DATETIME)
    FROM `pgd-prod-data-analytics.datalake.pgd_tbl_kredit`
    WHERE status_rek = '4'

    UNION all

    SELECT DISTINCT 
        t.cif, 
        t.branch_code, 
        CAST(NULL AS DATETIME) AS create_date -- FIX: Changed '' to NULL (Cast as DATETIME)
    FROM `pgd-prod-data-analytics.datalake.pgd_tbl_rekening_emas` e
    LEFT JOIN `pgd-prod-data-analytics.datalake.pgd_tbl_tabungan` t
        ON e.norek = t.norek
    WHERE e.saldo_akhir > 0
)
SELECT
    c.is_active,
    c.tipe_customer,
    c.flag,
    CAST(c.create_date AS TIMESTAMP)        AS create_date, 
    c.cif,
    c.nama,
    c.no_id                                 AS ktp,
    c.telp,
    c.hp,
    c.branch_code,
    CAST(c.update_date AS TIMESTAMP)        AS update_date,
    t.norek                                 AS nomor_rekening_tab_emas
FROM `pgd-prod-data-analytics.datalake.pgd_tbl_customer` c
LEFT JOIN aktif_cif a
    ON c.cif = a.cif
LEFT JOIN `pgd-prod-data-analytics.datalake.pgd_tbl_tabungan` t
    ON c.cif = t.cif
WHERE
    a.cif IS NOT NULL
    AND c.is_active = '1'
"""


In [26]:
DEST_TABLE = "pgd-dev-data-analytics.datamart_audit.dm_det_tbl_customer_fds"    

In [27]:
sql.execute_ddl(f"""create or replace table {DEST_TABLE} as 
{SELECT_QUERY}
""")
print(sql.read(f"select * from {DEST_TABLE} limit 10"))
print(sql.read(f"select count(*) from {DEST_TABLE} limit 10"))

[DDL] Executing: create or replace table pgd-dev-data-analytics.datamart_audit.dm_det_tbl_customer_fds as 

WITH aktif_cif AS (
    SELEC...
[DDL] Success.
[READ] Executing query...
[READ] Success → 10 row(s) returned.
  is_active tipe_customer    flag                      create_date  \
0         1             1  KONVEN        2019-07-25 16:32:31+00:00   
1         1             1  KONVEN 2019-06-03 03:20:23.198000+00:00   
2         1             1  KONVEN        2019-07-13 22:00:07+00:00   
3         1             1  KONVEN        2019-06-11 17:56:00+00:00   
4         1             1  KONVEN        2019-03-05 16:09:46+00:00   
5         1             1  KONVEN 2019-07-14 10:04:12.172000+00:00   
6         1             1  KONVEN        2019-07-25 16:32:31+00:00   
7         1             1  KONVEN        2019-05-08 21:49:23+00:00   
8         1             1  KONVEN 2019-07-27 12:08:49.022000+00:00   
9         1             1  KONVEN 2019-03-31 12:11:44.233000+00:00   

          